# Dynamic ORB Strategy
A stratey that chooses the best ORB TF to trade utilizing forest tree classifiers and time series

## Module.1
### Dependencies

In [17]:
from pathlib import Path
from typing import Optional

import sklearn as sk
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf


## Module.2
### Data importing and resampling

In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# Load 1-min NQ OHLCV + resample to 3m / 5m / 15m / 30m / 45m / 1h
# ─────────────────────────────────────────────────────────────────────────────

FILEPATH = "/Users/andrew/main/trading/data/NQ.1m.OHLCV.data/NQ-2010.06.06-2026.01.23-ohlcv-1m.csv"
TZ = "America/New_York"


# ── Loader ───────────────────────────────────────────────────────────────────
def load_ohlcv(filepath: str | Path, tz: Optional[str] = None) -> pd.DataFrame:
    """Load a local OHLCV CSV into a tidy, datetime-indexed DataFrame."""
    filepath = Path(filepath)
    if not filepath.exists():
        raise FileNotFoundError(filepath)

    raw = pd.read_csv(filepath)
    raw = raw.loc[:, ~raw.columns.str.match(r"^Unnamed: \d+$")]
    lower = {c.lower(): c for c in raw.columns}

    # Resolve datetime column
    dt_candidates = ["datetime", "date_time", "timestamp", "ts_event", "ts_recv", "ts", "time"]
    dt_col = next((lower[c] for c in dt_candidates if c in lower), None)

    if dt_col is None:
        date_col = lower.get("date")
        time_col = lower.get("time")
        if date_col and time_col:
            raw["__dt__"] = raw[date_col].astype(str) + " " + raw[time_col].astype(str)
            dt_col = "__dt__"
        elif date_col:
            dt_col = date_col
        else:
            raise ValueError(f"No datetime column found. Headers: {list(raw.columns)}")

    # Resolve OHLCV columns
    def pick(names):
        return next((lower[n] for n in names if n in lower), None)

    cols = {
        "open":   pick(["open", "o"]),
        "high":   pick(["high", "h"]),
        "low":    pick(["low", "l"]),
        "close":  pick(["close", "c", "last", "price"]),
        "volume": pick(["volume", "vol", "v", "size"]),
    }
    if any(v is None for v in cols.values()):
        raise ValueError(f"Unresolved columns: {cols}. Headers: {list(raw.columns)}")

    df = raw[[dt_col, *cols.values()]].copy()
    df.columns = ["datetime", *cols.keys()]

    # Datetime
    dt = pd.to_datetime(df["datetime"], utc=False, errors="coerce")
    if getattr(dt.dt, "tz", None) is None and tz:
        dt = dt.dt.tz_localize(tz, ambiguous="NaT", nonexistent="NaT")
    elif getattr(dt.dt, "tz", None) is not None and tz:
        dt = dt.dt.tz_convert(tz)

    df["datetime"] = dt
    df = df.dropna(subset=["datetime"]).set_index("datetime").sort_index()

    # Numerics
    for c in ("open", "high", "low", "close", "volume"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["open", "high", "low", "close"])

    return df


# ── Resampler ────────────────────────────────────────────────────────────────
AGG = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}


def resample_ohlcv(df: pd.DataFrame, rule: str, origin: str = "start_day") -> pd.DataFrame:
    """Resample a DatetimeIndexed OHLCV frame to a coarser frequency."""
    out = (
        df.resample(rule, label="left", closed="left", origin=origin)
          .agg(AGG)
          .dropna(subset=["open", "high", "low", "close"])
    )
    out = out[out["volume"].notna()]
    return out


# ── Load 1-minute ────────────────────────────────────────────────────────────
df_1m = load_ohlcv(FILEPATH, tz=TZ)
print(f"1m bars: {len(df_1m):,}")
print(f"Range:   {df_1m.index.min()}  →  {df_1m.index.max()}")
print(f"Cols:    {list(df_1m.columns)}")


# ── Build all resampled frames ───────────────────────────────────────────────
TIMEFRAMES = {
    "3m":  "3min",
    "5m":  "5min",
    "15m": "15min",
    "30m": "30min",
    "45m": "45min",
    "1h":  "1h",
}

frames: dict[str, pd.DataFrame] = {"1m": df_1m}

print()
for name, rule in TIMEFRAMES.items():
    print(f"resampling → {name} ({rule}) ...", end=" ")
    try:
        d = resample_ohlcv(df_1m, rule)
        frames[name] = d
        print(f"ok  rows={len(d):,}")
    except Exception as e:
        print(f"FAILED: {type(e).__name__}: {e}")
        raise

print("\nAll frames built:", list(frames.keys()))


# ── Display each resampled frame ─────────────────────────────────────────────
for name, d in frames.items():
    print("=" * 70)
    print(f"[{name}]  rows={len(d):,}   {d.index.min()}  →  {d.index.max()}")
    print("=" * 70)
    display(d.head(3))
    display(d.tail(3))
    display(d.describe())


# ── Summary table + volume conservation check ────────────────────────────────
summary = pd.DataFrame({
    "timeframe":    list(frames.keys()),
    "bars":         [len(d) for d in frames.values()],
    "start":        [d.index.min() for d in frames.values()],
    "end":          [d.index.max() for d in frames.values()],
    "first_close":  [d["close"].iloc[0]  for d in frames.values()],
    "last_close":   [d["close"].iloc[-1] for d in frames.values()],
    "total_volume": [int(d["volume"].sum()) for d in frames.values()],
})
display(summary)

ref = int(frames["1m"]["volume"].sum())
for name, d in frames.items():
    got = int(d["volume"].sum())
    assert got == ref, f"volume mismatch at {name}: {got:,} vs {ref:,}"
print(f"✓ volume conserved across all timeframes: {ref:,}")

1m bars: 6,949,239
Range:   2010-06-06 18:00:00-04:00  →  2026-01-23 16:59:00-05:00
Cols:    ['open', 'high', 'low', 'close', 'volume']

resampling → 3m (3min) ... ok  rows=1,821,013
resampling → 5m (5min) ... ok  rows=1,097,034
resampling → 15m (15min) ... ok  rows=366,257
resampling → 30m (30min) ... ok  rows=184,862
resampling → 45m (45min) ... ok  rows=124,477
resampling → 1h (1h) ... ok  rows=93,140

All frames built: ['1m', '3m', '5m', '15m', '30m', '45m', '1h']
[1m]  rows=6,949,239   2010-06-06 18:00:00-04:00  →  2026-01-23 16:59:00-05:00


,open,high,low,close,volume
datetime,,,,,
2010-06-06 18:00:00-04:00,1831.75,1831.75,1828.75,1829.00,117
2010-06-06 18:00:00-04:00,1833.50,1833.50,1827.75,1827.75,3
2010-06-06 18:01:00-04:00,1828.50,1831.00,1828.50,1830.50,99


,open,high,low,close,volume
datetime,,,,,
2026-01-23 16:57:00-05:00,25683.50,25685.0,25680.00,25682.25,125
2026-01-23 16:58:00-05:00,25683.00,25683.5,25678.50,25679.00,152
2026-01-23 16:59:00-05:00,25678.75,25680.5,25677.25,25680.50,191


,open,high,low,close,volume
count,6.949239e+06,6.949239e+06,6.949239e+06,6.949239e+06,6.949239e+06
mean,8.710671e+03,8.712190e+03,8.709134e+03,8.710670e+03,2.391639e+02
std,6.641935e+03,6.643037e+03,6.640804e+03,6.641931e+03,5.011294e+02
min,-5.495000e+01,-5.495000e+01,-5.495000e+01,-5.495000e+01,1.000000e+00
25%,3.464750e+03,3.465000e+03,3.464250e+03,3.464750e+03,9.000000e+00
50%,6.807500e+03,6.808750e+03,6.806000e+03,6.807500e+03,4.900000e+01
75%,1.344825e+04,1.345100e+04,1.344550e+04,1.344825e+04,2.150000e+02
max,2.680500e+04,2.680500e+04,2.680500e+04,2.680500e+04,2.036800e+04


[3m]  rows=1,821,013   2010-06-06 18:00:00-04:00  →  2026-01-23 16:57:00-05:00


,open,high,low,close,volume
datetime,,,,,
2010-06-06 18:00:00-04:00,1831.75,1833.5,1827.75,1831.25,251
2010-06-06 18:03:00-04:00,1828.00,1831.5,1828.00,1830.50,14
2010-06-06 18:06:00-04:00,1830.25,1831.0,1829.50,1829.50,15


... (20 outputs left out)

,open,high,low,close,volume
datetime,,,,,
2026-01-23 14:00:00-05:00,25726.25,25989.5,231.35,25748.0,31759
2026-01-23 15:00:00-05:00,25969.00,25982.0,230.80,25982.0,53841
2026-01-23 16:00:00-05:00,25974.00,25974.0,231.00,25680.5,14049


,open,high,low,close,volume
count,93140.000000,93140.000000,93140.000000,93140.000000,93140.000000
mean,8298.759022,8813.792120,3198.992204,8278.709155,17844.183884
std,6404.996739,6421.714846,4724.588615,6383.095042,26052.978626
min,-25.500000,25.700000,-54.950000,-26.850000,1.000000
25%,3193.250000,3629.437500,22.950000,3193.187500,1973.000000
50%,5948.375000,6655.000000,2034.875000,5943.500000,5752.000000
75%,12862.562500,13280.812500,4411.250000,12831.875000,22961.000000
max,26620.000000,26805.000000,26260.250000,26549.500000,248948.000000


,timeframe,bars,start,end,first_close,last_close,total_volume
0,1m,6949239,2010-06-06 18:00:00-04:00,2026-01-23 16:59:00-05:00,1829.00,25680.5,1662007287
1,3m,1821013,2010-06-06 18:00:00-04:00,2026-01-23 16:57:00-05:00,1831.25,25680.5,1662007287
2,5m,1097034,2010-06-06 18:00:00-04:00,2026-01-23 16:55:00-05:00,1830.50,25680.5,1662007287
3,15m,366257,2010-06-06 18:00:00-04:00,2026-01-23 16:45:00-05:00,1828.50,25680.5,1662007287
4,30m,184862,2010-06-06 18:00:00-04:00,2026-01-23 16:30:00-05:00,1828.75,25680.5,1662007287
5,45m,124477,2010-06-06 18:00:00-04:00,2026-01-23 16:15:00-05:00,1830.25,25680.5,1662007287
6,1h,93140,2010-06-06 18:00:00-04:00,2026-01-23 16:00:00-05:00,1820.00,25680.5,1662007287


✓ volume conserved across all timeframes: 1,662,007,287


In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# Vectorized: batch-download 720d of 1h yfinance data for 12 assets,
# then upsample to 1m / 3m / 5m / 15m / 30m / 45m.
# DST-safe: all bucket math done in UTC, converted back for display.
# ─────────────────────────────────────────────────────────────────────────────

SYMBOLS: dict[str, str] = {
    "YM":   "YM=F",
    "ES":   "ES=F",
    "RTY":  "RTY=F",
    "GC":   "GC=F",
    "SL":   "SI=F",       # Yahoo's silver ticker is SI=F
    "BTC":  "BTC-USD",
    "ETH":  "ETH-USD",
    "TSLA": "TSLA",
    "MSFT": "MSFT",
    "AAPL": "AAPL",
    "GOOG": "GOOG",
    "META": "META",
}

PERIOD   = "720d"
INTERVAL = "1h"

TARGETS: dict[str, str] = {
    "1m":  "1min",
    "3m":  "3min",
    "5m":  "5min",
    "15m": "15min",
    "30m": "30min",
    "45m": "45min",
}

OHLCV = ["open", "high", "low", "close", "volume"]

# ── Batch download (single batched call) ─────────────────────────────────────
tickers = list(SYMBOLS.values())
print(f"Downloading {len(tickers)} tickers in one batched call ...")

raw = yf.download(
    tickers=tickers,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=True,
    progress=False,
    group_by="ticker",     # → MultiIndex (ticker, field)
    threads=True,
    ignore_tz=False,
)

# yfinance returns flat columns if only one ticker resolved
if not isinstance(raw.columns, pd.MultiIndex):
    raw.columns = pd.MultiIndex.from_product(
        [[tickers[0]], raw.columns], names=["ticker", "field"]
    )

# ── Slice into per-label frames ──────────────────────────────────────────────
hourly: dict[str, pd.DataFrame] = {}

for label, ticker in SYMBOLS.items():
    if ticker not in raw.columns.get_level_values(0):
        print(f"[{label}] {ticker}: not present in download result")
        continue

    df = raw[ticker].copy()
    df.columns = [str(c).lower() for c in df.columns]
    keep = [c for c in OHLCV if c in df.columns]
    if len(keep) < 4:
        print(f"[{label}] {ticker}: missing OHLC columns — got {list(df.columns)}")
        continue

    df = df[keep]
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, utc=True)
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")

    df = df[~df.index.duplicated(keep="last")].sort_index()
    df = df.dropna(subset=["open", "high", "low", "close"])

    if df.empty:
        print(f"[{label}] {ticker}: empty after cleaning")
        continue

    hourly[label] = df
    print(f"[{label}] {ticker}: {len(df):,} bars  "
          f"{df.index.min()} → {df.index.max()}")

print(f"\nResolved {len(hourly)}/{len(SYMBOLS)} assets.\n")


# ── Vectorized, DST-safe upsampler ───────────────────────────────────────────
def upsample_ohlcv(df: pd.DataFrame, rule: str) -> pd.DataFrame:
    """
    Vectorized upsample of an hourly OHLCV frame to a finer timeframe.

    All bucket math is done in UTC to avoid DST ambiguity; the output index
    is converted back to the source timezone.
    """
    src_tz = df.index.tz

    # Build the fine grid in UTC so DST transitions can't raise
    start_utc = df.index.min().tz_convert("UTC")
    end_utc   = df.index.max().tz_convert("UTC")
    idx_utc   = pd.date_range(start_utc, end_utc, freq=rule, tz="UTC")
    idx       = idx_utc.tz_convert(src_tz)

    # 1) Forward-fill OHLC on the fine grid (single C-level pass)
    out = df.reindex(idx, method="ffill")

    # 2) Sub-bar counts per hour, computed in UTC (no ambiguity)
    hours = idx_utc.floor("h").values
    _, inverse, counts = np.unique(hours, return_inverse=True, return_counts=True)
    sub_per_hour = counts[inverse].astype(np.float64)

    # 3) Broadcast hourly volume down, divide by sub-bar count
    vol_hourly = df["volume"].reindex(idx, method="ffill").to_numpy(dtype=np.float64)
    out["volume"] = vol_hourly / sub_per_hour

    # 4) Trim leading edge before first real bar
    out = out.dropna(subset=["open", "high", "low", "close"])
    return out


# ── Build nested dict: frames[asset][timeframe] = DataFrame ──────────────────
frames: dict[str, dict[str, pd.DataFrame]] = {}

for label, h_df in hourly.items():
    frames[label] = {"1h": h_df}
    for tf_name, rule in TARGETS.items():
        frames[label][tf_name] = upsample_ohlcv(h_df, rule)

# ── Report ───────────────────────────────────────────────────────────────────
print("=" * 80)
print("ASSET × TIMEFRAME INVENTORY")
print("=" * 80)

for label, tfs in frames.items():
    print(f"\n{label}")
    for tf_name, d in tfs.items():
        print(f"    {tf_name:>3}  bars={len(d):>9,}   "
              f"{d.index.min()} → {d.index.max()}")

# ── Summary table ────────────────────────────────────────────────────────────
summary = pd.DataFrame(
    [
        {
            "asset":      label,
            "timeframe":  tf_name,
            "bars":       len(d),
            "start":      d.index.min(),
            "end":        d.index.max(),
            "last_close": d["close"].iloc[-1] if len(d) else np.nan,
        }
        for label, tfs in frames.items()
        for tf_name, d in tfs.items()
    ]
).sort_values(["asset", "timeframe"])
display(summary)

# ── Sanity: 1m bar count ≈ 60× the 1h bar count ─────────────────────────────
print("\nSanity check — 1m bars should be ≈60× the 1h bar count:")
for label, tfs in frames.items():
    h = len(tfs["1h"])
    m = len(tfs["1m"])
    ratio = m / h if h else 0
    print(f"  {label:>5}  1h={h:>6,}   1m={m:>8,}   ratio={ratio:>6.1f}")

# ── Volume conservation check ───────────────────────────────────────────────
print("\nVolume conservation check (1h sum vs 1m sum, per asset):")
for label, tfs in frames.items():
    v_h = float(tfs["1h"]["volume"].sum())
    v_m = float(tfs["1m"]["volume"].sum())
    ok = np.isclose(v_h, v_m, rtol=1e-6, atol=1e-3)
    print(f"  {label:>5}  1h={v_h:>18,.2f}   1m={v_m:>18,.2f}   {'✓' if ok else '✗'}")

[YM] YM=F: 13,483 bars  2024-05-12 18:00:00-04:00 → 2026-09-22 11:00:00-04:00
[ES] ES=F: 13,488 bars  2024-05-12 18:00:00-04:00 → 2026-09-22 11:00:00-04:00
[RTY] RTY=F: 13,513 bars  2024-05-12 18:00:00-04:00 → 2026-09-22 11:00:00-04:00
[GC] GC=F: 13,525 bars  2024-05-12 18:00:00-04:00 → 2026-09-22 11:00:00-04:00
[SL] SI=F: 13,525 bars  2024-05-12 18:00:00-04:00 → 2026-09-22 11:00:00-04:00
[BTC] BTC-USD: 17,239 bars  2024-10-02 20:00:00-04:00 → 2026-09-22 11:00:00-04:00
[ETH] ETH-USD: 17,236 bars  2024-10-02 20:00:00-04:00 → 2026-09-22 11:00:00-04:00
[TSLA] TSLA: 4,999 bars  2023-11-07 09:30:00-05:00 → 2026-09-22 11:30:00-04:00
[MSFT] MSFT: 4,999 bars  2023-11-07 09:30:00-05:00 → 2026-09-22 11:30:00-04:00
[AAPL] AAPL: 4,999 bars  2023-11-07 09:30:00-05:00 → 2026-09-22 11:30:00-04:00
[GOOG] GOOG: 4,999 bars  2023-11-07 09:30:00-05:00 → 2026-09-22 11:30:00-04:00
[META] META: 4,999 bars  2023-11-07 09:30:00-05:00 → 2026-09-22 11:30:00-04:00

Resolved 12/12 assets.

ASSET × TIMEFRAME INVENT

,asset,timeframe,bars,start,end,last_close
67,AAPL,15m,100805,2023-11-07 09:30:00-05:00,2026-09-22 11:30:00-04:00,343.220001
63,AAPL,1h,4999,2023-11-07 09:30:00-05:00,2026-09-22 11:30:00-04:00,343.220001
64,AAPL,1m,1512061,2023-11-07 09:30:00-05:00,2026-09-22 11:30:00-04:00,343.220001
68,AAPL,30m,50403,2023-11-07 09:30:00-05:00,2026-09-22 11:30:00-04:00,343.220001
65,AAPL,3m,504021,2023-11-07 09:30:00-05:00,2026-09-22 11:30:00-04:00,343.220001
...,...,...,...,...,...,...
1,YM,1m,1242301,2024-05-12 18:00:00-04:00,2026-09-22 11:00:00-04:00,52136.000000
5,YM,30m,41411,2024-05-12 18:00:00-04:00,2026-09-22 11:00:00-04:00,52136.000000
2,YM,3m,414101,2024-05-12 18:00:00-04:00,2026-09-22 11:00:00-04:00,52136.000000
6,YM,45m,27607,2024-05-12 18:00:00-04:00,2026-09-22 10:30:00-04:00,52213.000000



Sanity check — 1m bars should be ≈60× the 1h bar count:
     YM  1h=13,483   1m=1,242,301   ratio=  92.1
     ES  1h=13,488   1m=1,242,301   ratio=  92.1
    RTY  1h=13,513   1m=1,242,301   ratio=  91.9
     GC  1h=13,525   1m=1,242,301   ratio=  91.9
     SL  1h=13,525   1m=1,242,301   ratio=  91.9
    BTC  1h=17,239   1m=1,036,261   ratio=  60.1
    ETH  1h=17,236   1m=1,036,261   ratio=  60.1
   TSLA  1h= 4,999   1m=1,512,061   ratio= 302.5
   MSFT  1h= 4,999   1m=1,512,061   ratio= 302.5
   AAPL  1h= 4,999   1m=1,512,061   ratio= 302.5
   GOOG  1h= 4,999   1m=1,512,061   ratio= 302.5
   META  1h= 4,999   1m=1,512,061   ratio= 302.5

Volume conservation check (1h sum vs 1m sum, per asset):
     YM  1h=     61,251,090.00   1m=    101,224,692.00   ✗
     ES  1h=    842,180,276.00   1m=  1,581,873,087.50   ✗
    RTY  1h=    111,205,359.00   1m=    182,449,400.00   ✗
     GC  1h=    109,748,006.00   1m=    133,529,094.50   ✗
     SL  1h=     39,484,620.00   1m=     48,181,549.00   ✗
  

## Module.3
### ORB Strategy implementation

In [20]:
# time frame variations

In [21]:
# entry/exit logic

In [22]:
# risk management

## Module.4
### Feature engineering

In [23]:
# time series implementation

In [24]:
# event filter

In [25]:
# volume profile (aggregated)

## Module.5
### ML/AI

In [26]:
# forest tree classifiers

In [27]:
# NN

## Module.6
### Backtesting

In [28]:
# rolling wfa

In [29]:
# random wfa

In [30]:
# t-test simulation

In [31]:
# monte carlo simulation

In [32]:
# regression testing (for randomness/noise)